# 04 · SVM, Random Forest, XGBoost y SVD

**Jorge Javier Palma Martínez · ML1 FIUNA · 2026**

Protocolo: `rectangular-masked-v3-2026-11-22`.

Se muestran candidatos, ajustes y algoritmo SVD. La comparación utiliza train/validación; test no participa en estas decisiones.

Este cuaderno contiene el código ejecutable de la etapa, dividido en celdas con explicación. Ejecutar de arriba hacia abajo. Los notebooks se recorren del 01 al 06; los ajustes y las curvas pueden tardar varios minutos.

## 1. Librerías y ubicación del proyecto

Se cargan las bibliotecas utilizadas y se localiza la carpeta raíz. La misma celda funciona al abrir el notebook desde la raíz o desde `notebooks`. No entrena ningún modelo.

In [1]:
from pathlib import Path
import sys, json, hashlib, shutil, time, warnings, copy
import numpy as np
import pandas as pd
import cv2
import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/workflow.py").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    cohen_kappa_score,
)
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, learning_curve
from sklearn.base import clone
from xgboost import XGBClassifier
from src.features import FEATURE_COUNT, PREPROCESSING_SPEC
from src.models import SVDSubspaceClassifier


## 2. Configuración y carpetas de salida

Se fija la semilla, el orden de las cinco etiquetas y las rutas de matrices, informes y modelos. Crear estas carpetas no modifica las radiografías originales.

In [2]:
DATA = ROOT / "data/processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
MODELS = ROOT / "models"
LABELS = ["Normal", "Dudosa", "Leve", "Moderada", "Severa"]
SEED = 42
PROTOCOL = "rectangular-masked-v3-2026-11-22"
for folder in [DATA, REPORTS, FIGURES, MODELS]:
    folder.mkdir(parents=True, exist_ok=True)
print("Proyecto:", ROOT)
print("Protocolo:", PROTOCOL)


Proyecto: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla
Protocolo: rectangular-masked-v3-2026-11-22


## 3. Guardar resultados y mostrar las figuras

La primera función exporta JSON evitando NaN/Infinity. La segunda guarda cada gráfico y lo muestra dentro del notebook, para que el documento conserve código y salida al verse en GitHub.

In [3]:
def save_json(path, value):
    def finite(item):
        if isinstance(item, dict):
            return {k: finite(v) for k, v in item.items()}
        if isinstance(item, (list, tuple)):
            return [finite(v) for v in item]
        if isinstance(item, np.generic):
            item = item.item()
        if isinstance(item, float) and not np.isfinite(item):
            return None
        return item

    Path(path).write_text(
        json.dumps(
            finite(value), ensure_ascii=False, indent=2, allow_nan=False, default=str
        ),
        encoding="utf-8",
    )


def figure(name):
    path = FIGURES / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    display(Image(filename=str(path)))


## 4. Leer las matrices y definir las métricas

El cargador rechaza matrices de otra versión. La función de evaluación calcula accuracy, balanced accuracy, F1 y precisión/recall macro. F1 macro es el criterio de selección; no se ajustan transformaciones con test.

In [4]:
def arrays():
    payload = np.load(DATA / "features_arrays.npz")
    if "protocol" not in payload or str(payload["protocol"]) != PROTOCOL:
        raise ValueError(
            "Matrices de otra versión; ejecutar antes el preprocesamiento."
        )
    return payload


def score(y, pred):
    return {
        "Accuracy (%)": round(accuracy_score(y, pred) * 100, 4),
        "Balanced Acc (%)": round(balanced_accuracy_score(y, pred) * 100, 4),
        "F1 Macro": round(f1_score(y, pred, average="macro", zero_division=0), 6),
        "F1 Weighted": round(f1_score(y, pred, average="weighted", zero_division=0), 6),
        "Precision Macro": round(
            precision_score(y, pred, average="macro", zero_division=0), 6
        ),
        "Recall Macro": round(
            recall_score(y, pred, average="macro", zero_division=0), 6
        ),
    }


a = arrays()
display({k: v.shape for k, v in a.items() if k.startswith("X_")})


{'X_train_tabular': (1103, 288),
 'X_train_flat': (1103, 2304),
 'X_val_tabular': (236, 288),
 'X_val_flat': (236, 2304),
 'X_test_tabular': (237, 288),
 'X_test_flat': (237, 2304)}

## 5. Bucle completo de entrenamiento y validación

Este es el código que recorre candidatos: toma train, ejecuta `fit`, predice validación, calcula métricas y guarda modelos y resultados. El escalador/selector de cada Pipeline se ajusta solo con train. La función no accede a las entradas de test.

In [5]:
def fit_candidates(specs, filename):
    a = arrays()
    rows = []
    fitted = {}
    for name, family, kind, model in specs:
        X = a[f"X_train_{kind}"]
        V = a[f"X_val_{kind}"]
        t = time.perf_counter()
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=FutureWarning)
            model.fit(X, a["y_train"])
        metrics = score(a["y_val"], model.predict(V))
        rows.append(
            dict(
                Modelo=name,
                family=family,
                input_kind=kind,
                **metrics,
                seconds=round(time.perf_counter() - t, 2),
            )
        )
        fitted[name] = model
        print(name, metrics["F1 Macro"], flush=True)
    table = pd.DataFrame(rows).sort_values("F1 Macro", ascending=False, kind="stable")
    save_json(REPORTS / filename, table.to_dict(orient="records"))
    joblib.dump(fitted, MODELS / (filename.replace(".json", "_candidates.joblib")))
    return table


## 6. Configuración de SVM y XGBoost

SVM usa RBF, gamma scale, pesos balanceados y semilla 42. XGBoost usa profundidad 4, learning rate 0,05 y submuestreo 0,8. En cinco clases hay cinco árboles por ronda.

In [6]:
def svc(C=10):
    return SVC(
        C=C, kernel="rbf", class_weight="balanced", probability=True, random_state=SEED
    )


def xgb(n=200):
    return XGBClassifier(
        n_estimators=n,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=2,
    )


## 7. Candidatos SVM

Se crean tres Pipelines con C=1,10,50. Cada uno contiene su propio StandardScaler, que se ajustará con train.

In [7]:
specs = []

for C in [1.0, 10.0, 50.0]:
    specs.append(
        (
            f"SVM RBF C={C:g}",
            "SVM",
            "tabular",
            Pipeline([("scale", StandardScaler()), ("clf", svc(C))]),
        )
    )


## 8. Candidatos de árboles y ensambles

Se agregan 100/200/300 rondas de XGBoost y cuatro configuraciones de Random Forest. El código explicita cantidad de árboles, hoja mínima y pesos del bosque.

In [8]:
for n in [100, 200, 300]:
    specs.append((f"XGBoost {n} rondas", "XGBoost", "tabular", xgb(n)))

for n, leaf in [(100, 1), (200, 1), (200, 2), (300, 2)]:
    specs.append(
        (
            f"Random Forest n={n} hoja={leaf}",
            "Random Forest",
            "tabular",
            RandomForestClassifier(
                n_estimators=n,
                min_samples_leaf=leaf,
                class_weight="balanced_subsample",
                n_jobs=2,
                random_state=SEED,
            ),
        )
    )

print("Candidatos:", [item[0] for item in specs])


Candidatos: ['SVM RBF C=1', 'SVM RBF C=10', 'SVM RBF C=50', 'XGBoost 100 rondas', 'XGBoost 200 rondas', 'XGBoost 300 rondas', 'Random Forest n=100 hoja=1', 'Random Forest n=200 hoja=1', 'Random Forest n=200 hoja=2', 'Random Forest n=300 hoja=2']


## 9. Entrenar SVM y ensambles

Se ajusta cada candidato y se mide F1 en validación. Esta es una celda de entrenamiento real y puede tardar.

In [9]:
table = fit_candidates(specs, "advanced_validation.json")

display(table)


SVM RBF C=1 0.629974
SVM RBF C=10 0.611277
SVM RBF C=50 0.613015
XGBoost 100 rondas 0.592011
XGBoost 200 rondas 0.617142
XGBoost 300 rondas 0.616885
Random Forest n=100 hoja=1 0.530875
Random Forest n=200 hoja=1 0.566662
Random Forest n=200 hoja=2 0.553305
Random Forest n=300 hoja=2 0.572851


,Modelo,family,input_kind,Accuracy (%),Balanced Acc (%),F1 Macro,F1 Weighted,Precision Macro,Recall Macro,seconds
0,SVM RBF C=1,SVM,tabular,64.4068,63.2627,0.629974,0.640820,0.635643,0.632627,0.58
4,XGBoost 200 rondas,XGBoost,tabular,64.4068,60.4372,0.617142,0.634151,0.657159,0.604372,10.54
5,XGBoost 300 rondas,XGBoost,tabular,64.8305,60.7580,0.616885,0.636708,0.653815,0.607580,13.85
2,SVM RBF C=50,SVM,tabular,63.1356,60.6825,0.613015,0.627764,0.627515,0.606825,0.66
1,SVM RBF C=10,SVM,tabular,63.1356,60.6785,0.611277,0.627305,0.622077,0.606785,0.65
3,XGBoost 100 rondas,XGBoost,tabular,63.5593,58.8926,0.592011,0.618023,0.635046,0.588926,5.09
9,Random Forest n=300 hoja=2,Random Forest,tabular,61.4407,57.1167,0.572851,0.599173,0.612850,0.571167,1.77
7,Random Forest n=200 hoja=1,Random Forest,tabular,61.8644,56.4414,0.566662,0.594966,0.615096,0.564414,1.31
8,Random Forest n=200 hoja=2,Random Forest,tabular,60.5932,56.3031,0.553305,0.582620,0.593924,0.563031,1.13
6,Random Forest n=100 hoja=1,Random Forest,tabular,57.6271,52.5220,0.530875,0.555944,0.578559,0.525220,0.67


## 10. Algoritmo completo del clasificador SVD

La clase centra por grado y obtiene una base SVD; decide por menor residuo de proyección. Se incluye completa para estudiar el algoritmo. La celda de entrenamiento importa esta misma implementación desde `src.models`, de modo que joblib puede cargarla después desde la API sin depender de una clase definida en un notebook.

```python
class SVDSubspaceClassifier(ClassifierMixin, BaseEstimator):
    """Assign the nearest class-specific affine subspace by residual norm."""

    def __init__(self, k_components=30, temperature=0.25):
        self.k_components = k_components
        self.temperature = temperature

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]
        self.means_, self.bases_ = {}, {}
        for cls in self.classes_:
            values = X[np.asarray(y) == cls]
            self.means_[cls] = values.mean(axis=0)
            basis, _, _ = np.linalg.svd(
                (values - self.means_[cls]).T, full_matrices=False
            )
            self.bases_[cls] = basis[:, : self.k_components]
        return self

    def residuals(self, X):
        output = []
        for cls in self.classes_:
            delta = np.asarray(X) - self.means_[cls]
            basis = self.bases_[cls][:, : self.k_components]
            output.append(np.linalg.norm(delta - (delta @ basis) @ basis.T, axis=1))
        return np.column_stack(output)

    def predict(self, X):
        return self.classes_[self.residuals(X).argmin(axis=1)]

    def predict_proba(self, X):
        distances = self.residuals(X)
        # Normalize each row independently, so a sample's output does not depend
        # on other samples present in the request. These scores are uncalibrated.
        scores = (
            -distances
            / (distances.mean(axis=1, keepdims=True) + 1e-9)
            / self.temperature
        )
        scores -= scores.max(axis=1, keepdims=True)
        exp = np.exp(scores)
        return exp / exp.sum(axis=1, keepdims=True)
```

## 11. Ajustar el escalado y las bases SVD

Se aprende el escalador con píxeles de train y solo se transforma validación. Se calculan bases de hasta 50 componentes por clase para reutilizarlas al probar rangos menores.

In [10]:
a = arrays()

scaler = StandardScaler()

X = scaler.fit_transform(a["X_train_flat"])

V = scaler.transform(a["X_val_flat"])

model = SVDSubspaceClassifier(50).fit(X, a["y_train"])


## 12. Comparar rangos del subespacio

Se evalúan k=5,10,20,30,50 sin ajustar con validación. La clase predicha minimiza la norma residual; las probabilidades softmax de este clasificador no son una calibración clínica.

In [11]:
rows = table.to_dict(orient="records")

saved = joblib.load(MODELS / "advanced_validation_candidates.joblib")

for k in [5, 10, 20, 30, 50]:
    model.k_components = k
    name = f"SVD k={k}"
    rows.append(
        dict(
            Modelo=name,
            family="SVD",
            input_kind="flat",
            **score(a["y_val"], model.predict(V)),
        )
    )
    saved[name] = Pipeline(
        [("scale", copy.deepcopy(scaler)), ("clf", copy.deepcopy(model))]
    )


## 13. Guardar la comparación completa

Se reúnen los quince candidatos avanzados, se ordenan por F1 de validación y se conservan los objetos ajustados para la selección posterior.

In [12]:
table = pd.DataFrame(rows).sort_values("F1 Macro", ascending=False, kind="stable")

save_json(REPORTS / "advanced_validation.json", table.to_dict(orient="records"))

joblib.dump(saved, MODELS / "advanced_validation_candidates.joblib")

display(table)


,Modelo,family,input_kind,Accuracy (%),Balanced Acc (%),F1 Macro,F1 Weighted,Precision Macro,Recall Macro,seconds
0,SVM RBF C=1,SVM,tabular,64.4068,63.2627,0.629974,0.640820,0.635643,0.632627,0.58
1,XGBoost 200 rondas,XGBoost,tabular,64.4068,60.4372,0.617142,0.634151,0.657159,0.604372,10.54
2,XGBoost 300 rondas,XGBoost,tabular,64.8305,60.7580,0.616885,0.636708,0.653815,0.607580,13.85
3,SVM RBF C=50,SVM,tabular,63.1356,60.6825,0.613015,0.627764,0.627515,0.606825,0.66
4,SVM RBF C=10,SVM,tabular,63.1356,60.6785,0.611277,0.627305,0.622077,0.606785,0.65
5,XGBoost 100 rondas,XGBoost,tabular,63.5593,58.8926,0.592011,0.618023,0.635046,0.588926,5.09
6,Random Forest n=300 hoja=2,Random Forest,tabular,61.4407,57.1167,0.572851,0.599173,0.612850,0.571167,1.77
7,Random Forest n=200 hoja=1,Random Forest,tabular,61.8644,56.4414,0.566662,0.594966,0.615096,0.564414,1.31
8,Random Forest n=200 hoja=2,Random Forest,tabular,60.5932,56.3031,0.553305,0.582620,0.593924,0.563031,1.13
9,Random Forest n=100 hoja=1,Random Forest,tabular,57.6271,52.5220,0.530875,0.555944,0.578559,0.525220,0.67
